In [15]:
from core.settings import settings
from core.clients import apify_client, apify_tiktok_actor
from apps.tiktok.models import Video, VideoStats, RawData
from core.database import SessionFactory
from psycopg.errors import UniqueViolation
from sqlalchemy import select
from sqlalchemy.exc import IntegrityError

In [3]:
session = SessionFactory()
session

In [17]:
await session.rollback()

sys:1: SAWarning: Session's state has been changed on a non-active transaction - this state will be discarded.


In [18]:
videos_urls = """https://www.tiktok.com/@sumalavia.7/video/7622090562602028306
https://www.tiktok.com/@andreody_coach/video/7622066906131991815
https://www.tiktok.com/@brendagonzalesr/video/7621945617077619989
https://www.tiktok.com/@mamiofertass/video/7622025788581334279
https://www.tiktok.com/@pauespinozao/video/7623206456636099860
https://www.tiktok.com/@kairycc_/video/7623203883661184277
https://www.tiktok.com/@sumalavia.7/video/7623203343002766610
https://www.tiktok.com/@andreody_coach/video/7623189687041117447
https://www.tiktok.com/@brendagonzalesr/video/7623059137261227284
https://www.tiktok.com/@mamiofertass/video/7623133210653838613"""
for url in videos_urls.splitlines():
    try:
        video = Video(url=url)
        session.add(video)
        await session.commit()
    except IntegrityError:
        await session.rollback()

In [19]:
stmt = select(Video)
videos = (await session.scalars(stmt)).all()
videos

[Video(id=1, url='https://www.tiktok.com/@sumalavia.7/video/7622090562602028306', last_stat_id=None),
 Video(id=3, url='https://www.tiktok.com/@brendagonzalesr/video/7621945617077619989', last_stat_id=None),
 Video(id=4, url='https://www.tiktok.com/@andreody_coach/video/7622066906131991815', last_stat_id=None),
 Video(id=5, url='https://www.tiktok.com/@mamiofertass/video/7622025788581334279', last_stat_id=None),
 Video(id=6, url='https://www.tiktok.com/@mamiofertass/video/7623133210653838613', last_stat_id=None),
 Video(id=7, url='https://www.tiktok.com/@brendagonzalesr/video/7623059137261227284', last_stat_id=None),
 Video(id=16, url='https://www.tiktok.com/@pauespinozao/video/7623206456636099860', last_stat_id=None),
 Video(id=17, url='https://www.tiktok.com/@kairycc_/video/7623203883661184277', last_stat_id=None),
 Video(id=18, url='https://www.tiktok.com/@sumalavia.7/video/7623203343002766610', last_stat_id=None),
 Video(id=19, url='https://www.tiktok.com/@andreody_coach/video/7623

In [20]:
run_input = {
  "postURLs": [video.url for video in videos],
}
run = await apify_tiktok_actor.call(run_input=run_input)

[apify.tiktok-scraper runId:x2L6Pf3F9aj3Ko0fx] -> Status: RUNNING, Message: 
[apify.tiktok-scraper runId:x2L6Pf3F9aj3Ko0fx] -> 2026-04-01T00:49:25.331Z ACTOR: Pulling container image of build LD87KmnAxsPdgtOEx from registry.
[apify.tiktok-scraper runId:x2L6Pf3F9aj3Ko0fx] -> 2026-04-01T00:49:25.333Z ACTOR: Creating container.
[apify.tiktok-scraper runId:x2L6Pf3F9aj3Ko0fx] -> 2026-04-01T00:49:25.398Z ACTOR: Starting container.
[apify.tiktok-scraper runId:x2L6Pf3F9aj3Ko0fx] -> 2026-04-01T00:49:25.400Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.tiktok-scraper runId:x2L6Pf3F9aj3Ko0fx] -> 2026-04-01T00:49:25.578Z Running on architecture: x86_64
[apify.tiktok-scraper runId:x2L6Pf3F9aj3Ko0fx] -> 2026-04-01T00:49:25.580Z Will run command: xvfb-run -a -s "-ac -screen 0 1920x1080x24+32 -nolisten tcp" /bin/bash -o pipefail -c bash -c "    java         --enable-native-access=ALL-UNNAMED         -cp classes-test:classes-main:lib/*         com.zhiliaoapp.musically.MetaSec    

In [21]:
async for item in apify_client.dataset(run["defaultDatasetId"]).iterate_items():
    url = item["webVideoUrl"]
    video = next(video for video in videos if video.url == item["webVideoUrl"])

    raw_data = RawData(data=item)
    session.add(raw_data)

    video_stats = VideoStats(
        video=video,
        raw_data=raw_data,
        diggs=item["diggCount"],
        shares=item["shareCount"],
        plays=item["playCount"],
        collects=item["collectCount"],
        comments=item["commentCount"],
        reposts=item["repostCount"],
    )
    session.add(video_stats)
    await session.commit()

In [16]:
video_stats = VideoStats(
    video=videos[0],
    raw_data=raw_data,
    diggs=item["diggCount"],
    shares=item["shareCount"],
    plays=item["playCount"],
    collects=item["collectCount"],
    comments=item["commentCount"],
    reposts=item["repostCount"],
)
session.add(video_stats)
await session.commit()

sys:1: SAWarning: Object of type <VideoStats> not in session, add operation along 'Video.stats' will not proceed


In [19]:
list(session)

[Video(id=1, url='https://www.tiktok.com/@sumalavia.7/video/7622090562602028306', last_stat_id=None),
 RawData(id=1, data={'id': '7622090562602028306', 'text': '#Publicidad \n\nSé que el ceviche te queda rico, pero con el Paiche de Yummei te quedará aún mejor. 🐟✨\n\n\n\nDisponible en @wongoficial , aporta una frescura y textura firme que marcan la diferencia. 🩵\n\n\n\nMantén la tradición en casa con calidad excepcional. 🛒\n\n\n\n@Blooper Strategy Partner \n\n#SemanaSanta #Ceviche #receta #creatorsearchinsights ', 'textLanguage': 'es', 'createTime': 1774656260, 'createTimeISO': '2026-03-28T00:04:20.000Z', 'locationCreated': 'PE', 'isAd': False, 'authorMeta': {'id': '7004858351788295174', 'name': 'sumalavia.7', 'profileUrl': 'https://www.tiktok.com/@sumalavia.7', 'nickName': 'Alberto Sumalavia', 'verified': False, 'signature': '🧑\u200d🍳 Creador gastronómico | Cocinero 🇵🇪', 'avatar': 'https://p16-common-sign.tiktokcdn-us.com/tos-alisg-avt-0068/0141375173e071ffe61bc4c696e94579~tplv-tiktokx